# OrthoDiffusion Effusion probe

Validates whether OrthoDiffusion (arXiv:2602.20752, real released weights -- not a reconstruction)
actually reproduces `prvsiyan/rsna-knee-two-teacher-bundle-v1`'s own reported gold-gate numbers
before trusting it for anything. That bundle's own honest, nested-CV receipt showed Effusion alone
at ridge_auc=0.9578 (vs. our existing blend's 0.9068) while most other targets got zero deployment
weight -- the opposite of the "ACL 5% / Effusion 50%" claim in two flashier, unverified notebooks.

Architecture and descriptor contract confirmed directly from the official OrthoDiffusion source
(not guessed): `create_model(256, 64, 1, in_channels=1, out_channels=1)` with all defaults
(channel_mult=(1,1,2,2,4) for image_size=256, attention_resolutions="16", num_heads=1), cosine beta
schedule (guided-diffusion default), `mid_2` block = 256 channels at the bottleneck. Pooling
(mean + max + std + adaptive-(1,2,2) pool = 256*(1+1+1+4) = 1792) matches the bundle's stated
1792-per-plane exactly -- this is a strong sign the reconstruction is on the right track, unlike
the `dino` attempt which never had an exact-match checkpoint like this to confirm against.

Two things are still genuinely guessed and could be wrong: (1) the 4 "deterministic" noise-draw
seeds (using 0,1,2,3), (2) the 3 extra input features beyond the 384 PCA dims (guessing a
per-plane presence mask). If the reproduced AUCs don't match the receipt's numbers, that tells us
which guess to revisit -- same discipline as the `dino` retry.


In [ ]:
import os, sys, json, hashlib, shutil
import numpy as np, pandas as pd, torch, torch.nn.functional as F
torch.set_grad_enabled(False)

# --- GPU probe: torch.cuda.is_available() lies on a broken P100 (established finding, see
# README.md). Force machine_shape=NvidiaTeslaT4 in kernel-metadata.json is not enough alone --
# verify with a real op. ---
DEV = 'cpu'
if torch.cuda.is_available():
    try:
        _t = torch.randn(4, 4, device='cuda') @ torch.randn(4, 4, device='cuda')
        _t.cpu()
        DEV = 'cuda'
    except Exception as e:
        print(f'[gpu-probe] CUDA claimed available but failed a real op ({e}); falling back to CPU')
print('device:', DEV, '|', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'cpu')

def _find_dir(name, root='/kaggle/input'):
    for r, dirs, files in os.walk(root):
        if os.path.basename(r) == name:
            return r
    print('--- /kaggle/input tree (debug) ---')
    for r, dirs, files in os.walk('/kaggle/input'):
        depth = r[len('/kaggle/input'):].count(os.sep)
        if depth <= 3:
            print(r)
    raise FileNotFoundError(f'{name} not found under {root}')

ROOT = os.path.dirname(_find_dir('diffusion_model'))
print('ROOT resolved to:', ROOT)
# The mirrored dataset may have dropped diffusion_model/__init__.py (files starting with "_" or
# empty files are sometimes filtered by dataset upload tooling) -- input is read-only, so copy to
# a writable location and add it back if missing.
WORK_PKG = '/kaggle/working/od_pkg'
shutil.copytree(os.path.join(ROOT, 'diffusion_model'), os.path.join(WORK_PKG, 'diffusion_model'), dirs_exist_ok=True)
open(os.path.join(WORK_PKG, 'diffusion_model', '__init__.py'), 'a').close()
sys.path.insert(0, WORK_PKG)
sys.path.insert(0, ROOT)
from diffusion_model.unet import create_model

def _find_file(name, root='/kaggle/input'):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(f'{name} not found under {root}')

PLANES = ['axial', 'coronal', 'sagittal']
WEIGHT_PATH = {p: _find_file(f'{p}_ema.pt') for p in PLANES}
print('weight paths:', WEIGHT_PATH)

def load_backbone(path):
    m = create_model(256, 64, 1, in_channels=1, out_channels=1).to(DEV).eval()
    ck = torch.load(path, map_location='cpu')
    sd = ck['ema'] if 'ema' in ck else ck
    new_sd = {}
    for k, v in sd.items():
        k2 = k.replace('denoise_fn.module.', '').replace('denoise_fn.', '')
        new_sd[k2] = v
    missing, unexpected = m.load_state_dict(new_sd, strict=False)
    print(f'  {os.path.basename(path)}: loaded, missing={len(missing)} unexpected={len(unexpected)}')
    assert len(missing) == 0, missing[:5]
    return m

MODELS = {p: load_backbone(WEIGHT_PATH[p]) for p in PLANES}

# cosine beta schedule (guided-diffusion default, s=0.008) -- matches GaussianDiffusion's own
# default when no betas= is passed, which is what both train.py and linear_classifier.py do.
def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = np.linspace(0, timesteps, steps)
    ac = np.cos(((x / timesteps) + s) / (1 + s) * np.pi * 0.5) ** 2
    ac = ac / ac[0]
    betas = 1 - (ac[1:] / ac[:-1])
    return np.clip(betas, 0, 0.999)

TIMESTEPS = 1000
betas = cosine_beta_schedule(TIMESTEPS)
alphas_cumprod = np.cumprod(1.0 - betas)
SQRT_AC = torch.tensor(np.sqrt(alphas_cumprod), dtype=torch.float32, device=DEV)
SQRT_1M_AC = torch.tensor(np.sqrt(1 - alphas_cumprod), dtype=torch.float32, device=DEV)
print('beta schedule ready, alphas_cumprod[100] =', alphas_cumprod[100])


In [ ]:
# --- descriptor extraction: mid_2 @ t=100, 4 deterministic noise draws x (forward + depth-reversed),
# pooled to 1792 = 256*(mean + max + std + adaptive(1,2,2)) and averaged over the 8 combinations. ---
T_IDX = torch.tensor([100], device=DEV)

def q_sample(x0, seed):
    g = torch.Generator(device='cpu').manual_seed(seed)
    noise = torch.randn(x0.shape, generator=g).to(DEV)
    return SQRT_AC[100] * x0 + SQRT_1M_AC[100] * noise

def pool_1792(feat):
    # feat: (1, C, D, H, W)
    mean_f = feat.mean(dim=(2, 3, 4))
    max_f = feat.amax(dim=(2, 3, 4))
    std_f = feat.std(dim=(2, 3, 4))
    adap_f = F.adaptive_avg_pool3d(feat, (1, 2, 2)).flatten(1)
    return torch.cat([mean_f, max_f, std_f, adap_f], dim=1)  # (1,1792)

def descriptor(model, vol):
    # vol: (16,256,256) float32 in [-1,1], single plane
    x0 = torch.from_numpy(vol)[None, None].to(DEV)  # (1,1,16,256,256)
    outs = []
    for orient in (x0, torch.flip(x0, dims=[2])):   # forward, depth-reversed
        for seed in range(4):
            xt = q_sample(orient, seed)
            _, info = model(xt, T_IDX, ret_mid=True)
            outs.append(pool_1792(info['mid']['mid_2']).float().cpu().numpy())
    return np.mean(np.stack(outs), axis=0)[0]  # (1792,)

print('descriptor() ready')


In [ ]:
# --- study selection + DICOM decode: one series per plane, prefer fluid-sensitive fat-suppressed,
# >=16 slices; central 16 slices, resize 256x256, min-max normalize per volume, then x/127.5-1. ---
import pydicom, cv2
cv2.setNumThreads(1)

COMP = os.path.dirname(_find_file('train.csv'))
tr = pd.read_csv(os.path.join(COMP, 'train.csv'))
ser = pd.read_csv(os.path.join(COMP, 'train_series.csv'))
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA",
       "Effusion","Synovitis","Baker's","Contusion","Fracture"]
gold = tr[tr[LAB].notna().all(axis=1)].copy()
print('gold studies:', len(gold))

PLANE_NAME = {'axial': 'Axial', 'coronal': 'Coronal', 'sagittal': 'Sagittal'}

def pick_series(g, plane):
    cand = g[g.Anatomical_Plane == PLANE_NAME[plane]].copy()
    if not len(cand):
        return None
    cand['score'] = (cand.get('Fluid_Sensitive', 0).fillna(0) * 2
                     + cand.get('Fat_Suppression', 0).fillna(0))
    cand = cand.sort_values('score', ascending=False)
    return cand.iloc[0].SeriesInstanceUID

def ordered_files(sdir):
    keyed = []
    for f in os.listdir(sdir):
        if not f.endswith('.dcm'):
            continue
        try:
            ds = pydicom.dcmread(os.path.join(sdir, f), stop_before_pixels=True)
            keyed.append((float(getattr(ds, 'InstanceNumber', 0)), os.path.join(sdir, f)))
        except Exception:
            pass
    return [p for _, p in sorted(keyed)]

def build_plane_volume(uid, plane):
    g = ser[ser.StudyInstanceUID == uid]
    sid = pick_series(g, plane)
    if sid is None:
        return None
    sdir = os.path.join(COMP, 'train_series', uid, str(sid))
    if not os.path.isdir(sdir):
        return None
    files = ordered_files(sdir)
    if len(files) < 3:
        return None
    n = len(files)
    if n >= 16:
        lo = (n - 16) // 2
        pick = files[lo:lo + 16]
    else:
        pick = files + [files[-1]] * (16 - n)
    imgs = []
    for p in pick:
        ds = pydicom.dcmread(p)
        a = ds.pixel_array.astype(np.float32)
        imgs.append(cv2.resize(a, (256, 256), interpolation=cv2.INTER_AREA))
    vol = np.stack(imgs)  # (16,256,256)
    lo, hi = vol.min(), vol.max()
    vol = (vol - lo) / max(hi - lo, 1e-6) * 255.0
    return (vol / 127.5 - 1.0).astype(np.float32)

print('build_plane_volume() ready')


In [ ]:
# --- run over the 58 gold studies, extract descriptors, apply PCA + ridge head, compare to the
# receipt's own reported per-target AUCs (the verification gate before trusting anything further). ---
from sklearn.metrics import roc_auc_score

AST = os.path.dirname(_find_file('teacher_ridge_head_bundle.npz'))
PCA_COMP = {p: np.load(f'{AST}/pca_{p}_components_f32.npy') for p in PLANES}   # (128,1792)
PCA_MEAN = {p: np.load(f'{AST}/pca_{p}_mean_f32.npy') for p in PLANES}          # (1792,)
head = np.load(f'{AST}/teacher_ridge_head_bundle.npz')
COEF, INTERCEPT = head['coef'], head['intercept']                 # (12,387) (12,)
SCALER_MEAN, SCALER_SCALE = head['scaler_mean'], head['scaler_scale']  # (12,387)

def pca_transform(x, plane):
    return (x - PCA_MEAN[plane]) @ PCA_COMP[plane].T   # (128,)

rows = []
t0 = __import__('time').time()
for i, r in enumerate(gold.itertuples()):
    uid = r.StudyInstanceUID
    plane_feats, presence = [], []
    for plane in PLANES:
        vol = build_plane_volume(uid, plane)
        if vol is None:
            plane_feats.append(np.zeros(128, np.float32))
            presence.append(0.0)
            continue
        desc = descriptor(MODELS[plane], vol)
        plane_feats.append(pca_transform(desc, plane))
        presence.append(1.0)
    x = np.concatenate(plane_feats + [np.array(presence, np.float32)])   # (387,)
    z = (x - SCALER_MEAN) / SCALER_SCALE          # (12,387) broadcast
    logit = (z * COEF).sum(axis=1) + INTERCEPT
    prob = 1 / (1 + np.exp(-logit))
    rows.append({'StudyInstanceUID': uid, 'presence': tuple(presence), **{LAB[j]: prob[j] for j in range(12)}})
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(gold)} | {__import__("time").time()-t0:.0f}s', flush=True)

pred = pd.DataFrame(rows).set_index('StudyInstanceUID')
gold_idx = gold.set_index('StudyInstanceUID')
print('\npresence rate (axial,coronal,sagittal):', np.mean([p for p in pred['presence']], axis=0))

RECEIPT_RIDGE_AUC = {  # from prvsiyan/rsna-knee-two-teacher-bundle-v1's own receipt.json
    'ACL': 0.6262, 'MCL': 0.5850, 'Medial Meniscus': 0.6046, 'Lateral Meniscus': 0.7205,
    'Medial OA': 0.7147, 'Lateral OA': 0.8143, 'PF OA': 0.7310, 'Effusion': 0.9578,
    'Synovitis': 0.6607, "Baker's": 0.7283, 'Contusion': 0.6140, 'Fracture': 0.6194,
}
print(f"\n{'target':18s} {'our_auc':>8s} {'receipt_auc':>12s} {'diff':>8s}")
diffs = []
for name in LAB:
    y = gold_idx[name].values
    p = pred[name].values
    auc = roc_auc_score(y, p) if len(set(y.astype(int))) > 1 else float('nan')
    d = auc - RECEIPT_RIDGE_AUC[name]
    diffs.append(d)
    print(f'{name:18s} {auc:8.4f} {RECEIPT_RIDGE_AUC[name]:12.4f} {d:+8.4f}')
print(f'\nmean |diff| = {np.nanmean(np.abs(diffs)):.4f}  (small = reconstruction matches; large = a guess above is wrong)')
pred.to_csv('/kaggle/working/orthodiffusion_gold_predictions.csv')
